# Colab Runner Template (standard practice)
Generic harness: run any repo job on Colab GPU/high-RAM instead of the Mac.
Fill the PARAMS cell, Runtime -> GPU if needed, Run All.
Used for: large data crunches, GPU-only models (e.g. Unlimited-OCR), long batch jobs.
Requirements are tied to the repo: this installs `requirements*.txt` (including
`requirements-ocr.txt` when present) so Colab mirrors the repo's declared deps.

In [ ]:
# PARAMS — edit these
REPO   = 'herrrickshaw/global-market-research-platform'  # public GitHub repo
BRANCH = 'main'
SUBDIR = ''            # optional working subdir inside the repo
CMD    = 'python3 --version'   # the job to run
SERVE_PORT = 0         # >0 = long-running server: expose via cloudflared tunnel

In [ ]:
!nvidia-smi -L || echo 'no GPU runtime'
import os
!git clone --depth 1 -b {BRANCH} https://github.com/{REPO}.git job
%cd job
if SUBDIR: os.chdir(SUBDIR)
# Install every requirements file the repo declares (incl. requirements-ocr.txt)
import glob, subprocess
for req in sorted(glob.glob('requirements*.txt')):
    print('installing', req)
    subprocess.run(['pip','-q','install','-r',req], check=False)

In [ ]:
import subprocess, time, requests, re
if SERVE_PORT:
    proc = subprocess.Popen(CMD, shell=True)
    for _ in range(180):
        time.sleep(10)
        try:
            if requests.get(f'http://localhost:{SERVE_PORT}', timeout=5).status_code < 500:
                print('server is up'); break
        except Exception: pass
    subprocess.run(['wget','-q','-O','cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    subprocess.run(['chmod','+x','cloudflared'])
    tun = subprocess.Popen(['./cloudflared','tunnel','--url',f'http://localhost:{SERVE_PORT}'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0=time.time(); url=None
    while time.time()-t0 < 60 and url is None:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tun.stdout.readline())
        if m: url = m.group(0)
    print('PUBLIC URL =', url)
else:
    !{CMD}

Batch jobs: write outputs into the cloned repo and push from Colab (needs a token), or download via the Files pane / Dropbox. Keep the tab open for servers — free Colab disconnects after a few hours.